## Glassdoor Job Postings: Data Cleaning and Feature Engineering
This notebook prepares a dataset containing 674 data-related job postings on Glassdoor. 
The workflow includes an initial data quality assessment, standardization of attributes, job-role classification, location parsing and extraction of required technical skills based on job descriptions.

### 1. Data loading and initial data quality assessment

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('jobs.csv')

In [3]:
quality_summary = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=False),
    "duplicate_values": df.duplicated().sum()
})

display(quality_summary)

,data_type,missing_values,unique_values,duplicate_values
index,int64,0,672,0
Job Title,object,0,172,0
Salary Estimate,object,0,30,0
Job Description,object,0,489,0
Rating,float64,0,32,0
Company Name,object,0,432,0
Location,object,0,207,0
Headquarters,object,0,229,0
Size,object,0,9,0
Founded,int64,0,103,0


Although there are no explicit missing values, certain fields in the data marked as "-1" or "Unknown" represent missing values. These are handled in the data cleaning steps further below.

### 2. Standardizing column names
All column names are converted to snake case.

In [4]:
df.columns = df.columns.str.replace(' ', '_').str.lower().str.strip()

### 3. Simplifying Job Titles
All job titles are simplified and assigned to one of four most common roles: Data Analyst, Data Engineer, Data Scientist and Machine Learning Engineer. All other job titles are classified as "Other".

In [5]:
def simplify_title(title):
    t = str(title).lower()

    if "machine learning" in t:
        return "Machine Learning Engineer"
    elif "data scientist" in t:
        return "Data Scientist"
    elif "data analyst" in t:
        return "Data Analyst"
    elif "data engineer" in t:
        return "Data Engineer"
    else:
        return "Other"

df['job_title'] = df['job_title'].apply(simplify_title)

### 4. Standardizing salary values
Minimum and maximum estimated salary values are extracted and converted to numbers. An average salary amount is calculated based on these values.

In [6]:
df['salary_estimate'] = df['salary_estimate'].str.replace('(Glassdoor est.)', '').str.replace('(Employer est.)', '').str.strip()
df['min_salary'] = df['salary_estimate'].str.split('-').str.get(0)
df['max_salary'] = df['salary_estimate'].str.split('-').str.get(1)


df['min_salary'] = df['min_salary'].str.replace('$', '').str.replace('K', '')
df['max_salary'] = df['max_salary'].str.replace('$', '').str.replace('K', '')


df['min_salary'] = df['min_salary'].astype(int)
df['max_salary'] = df['max_salary'].astype(int)


df['min_salary'] = df['min_salary'] * 1000
df['max_salary'] = df['max_salary'] * 1000
df["avg_salary"] = (df["min_salary"] + df["max_salary"]) / 2

df = df.drop(columns='salary_estimate')

### 5. Standardizing location values
Each location is split into two new attributes: City and State.

In [7]:
state_mapping = pd.read_excel('state_mapping.xlsx')

state_mapping = state_mapping.rename(columns={
    'Full Name': 'state_name',
    '2-letter USPS': 'abbreviation'
})

state_abbr = state_mapping.set_index('state_name')['abbreviation'].to_dict()

def clean_location(loc):
    if pd.isna(loc):
        return 'Unknown', 'Unknown'

    x = str(loc).strip()

    if x.lower() == 'remote':
        return 'Remote', 'Remote'

    if ',' in x:
        parts = [p.strip() for p in x.split(',')]
        if len(parts) == 3:
            city = parts[0]
            state = parts[2]
            return city, state 
        if len(parts) == 2:
            city, state = parts 
            return city, state
        
    if x.title() in state_abbr:
        return 'Unknown', state_abbr[x.title()]

    return 'Unknown', 'Unknown'

df[['city', 'state']] = df['location'].apply(
    lambda loc: pd.Series(clean_location(loc))
)

df = df.drop(columns='location')

### 6. Cleaning Company, Industry, Sector and Ownership values

In [8]:
df['company_name'] = df['company_name'].str.replace(r'\s*\d\.\d\s*$', '', regex=True).str.strip()

df['industry'] = df['industry'].str.replace('-1', 'Unknown').str.strip()
df['sector'] = df['sector'].str.replace('-1', 'Unknown').str.strip()
df['type_of_ownership'] = df['type_of_ownership'].str.replace('-1', 'Unknown').str.strip()

### 7. Extracting job skills based on the description

In [9]:
skill_patterns = {
    "sql": r"\bsql\b|\bt-sql\b|\bmysql\b|\bpostgresql\b",
    "python": r"\bpython\b",
    "excel": r"\bexcel\b|\bmicrosoft excel\b",
    "power_bi": r"\bpower\s*bi\b|\bpowerbi\b",
    "tableau": r"\btableau\b",
    "r": r"\br programming\b|\br language\b",
    "aws": r"\baws\b|\bamazon web services\b",
    "azure": r"\bazure\b",
    "gcp": r"\bgcp\b|\bgoogle cloud\b",
    "spark": r"\bapache spark\b|\bpyspark\b",
    "snowflake": r"\bsnowflake\b",
    "databricks": r"\bdatabricks\b",
}

description = df["job_description"].fillna("")

for skill, pattern in skill_patterns.items():
    df[f"skill_{skill}"] = (
        description
        .str.contains(pattern, case=False, regex=True, na=False)
        .astype("int8")
    )

### 8. Cleaning and standardizing company size values


In [10]:
df['size'] = df['size'].str.replace('-1', 'Unknown').str.replace(' to ', '-').str.replace('employees', ' ').str.strip()

### 9. Dropping attributes outside of the scope of the analysis

In [11]:
df = df.drop(columns = ['job_description', 'headquarters', 'founded', 'competitors', 'index'])

In [12]:
df.head()

,job_title,rating,company_name,size,type_of_ownership,industry,sector,revenue,min_salary,max_salary,...,skill_excel,skill_power_bi,skill_tableau,skill_r,skill_aws,skill_azure,skill_gcp,skill_spark,skill_snowflake,skill_databricks
0,Data Scientist,3.1,Healthfirst,1001-5000,Nonprofit Organization,Insurance Carriers,Insurance,Unknown / Non-Applicable,137000,171000,...,0,0,0,0,1,1,0,0,0,0
1,Data Scientist,4.2,ManTech,5001-10000,Company - Public,Research & Development,Business Services,$1 to $2 billion (USD),137000,171000,...,0,0,0,0,0,0,0,0,0,0
2,Data Scientist,3.8,Analysis Group,1001-5000,Private Practice / Firm,Consulting,Business Services,$100 to $500 million (USD),137000,171000,...,0,0,0,0,1,0,0,0,0,0
3,Data Scientist,3.5,INFICON,501-1000,Company - Public,Electrical & Electronic Manufacturing,Manufacturing,$100 to $500 million (USD),137000,171000,...,0,0,0,0,0,0,0,0,0,0
4,Data Scientist,2.9,Affinity Solutions,51-200,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,137000,171000,...,0,0,0,0,0,0,0,0,0,0


### 10. Exporting the cleaned dataset

In [13]:
df.to_csv("jobs_cleaned.csv", index=False)